# 1 — Data Summary

## Dataset Overview
- What is the original format, structure, and approximate size of the full dataset?
- Describe the features (columns) across the selected tables.
- Provide a statistical summary of numerical columns, including:
  - Counts  
  - Distributions  
  - Ranges  

## Data Cleaning Methodology
- How will errors, null values, duplicates, and outliers be handled?
- What data quality standards will be enforced for the **Silver layer**?
- Does the dataset require augmentation with external sources?
  - If so, specify which sources and explain why they are needed.

---

# 2 — Proposed Solution

## Silver Layer Design
- Provide a complete **Entity Relationship Diagram (ERD)** representing the normalized Silver layer schema.

## Gold Layer Design
- Describe the Gold layer structure:
  - What tables or views exist?
  - How is the data denormalized?
  - How does the design support the analytical use case?

## Storage & Organization
- Describe the file and directory structure for:
  - Bronze layer
  - Silver layer
- Specify the **partitioning strategy** for the dataset (e.g., by date, port, year, etc.).

---

# 3 — Use Case

## Analytical Use Case Definition
- Choose or define an analytical use case for the Gold layer.

### Default Use Case
Provide aggregated analysis on shipping activity for major U.S. ports, segmented by:
- Type of goods  
- Country of origin  
- Carrier  
- Time period  

### Requirements
- Use at least **3 of the 10 source tables**.
- Include data from **more than one year**, unless otherwise approved by the instructor.

## 1 — Data Summary

### Dataset Overview

#### Format & Structure
- Data is stored as **CSV files** in a **year‑partitioned directory structure**
- Each year contains consistent subject‑area tables
- The dataset follows a **relational data model**

#### Tables Used (3)
- **header** — Shipment‑level metadata
- **container** — Physical container details
- **hazmatclass** — Hazardous material classifications

---

### Feature Summary (Selected Tables)

#### header — Shipment Context
- **Keys:** identifier
- **Routing:** foreign port, U.S. port of unlading/destination
- **Carrier/Vessel:** carrier_code, vessel_name
- **Timing:** estimated and actual arrival dates
- **Status:** record lifecycle (New / Amended / Deleted)
- **Measures:** manifest quantity, weight, measurement

#### container — Physical Containers
- **Keys:** identifier, container_number
- **Dimensions:** container length, height, width
- **Type & Status:** container type, load status, service type
- **Security:** seal numbers

#### hazmatclass — Hazard Classification
- **Keys:** identifier, container_number, hazmat_sequence_number
- **Classification:** IMDG/UN hazard class (e.g., 3, 6.1, 8)


In [14]:
# Statistical Summary of My Data
import pandas as pd
# Load the several files into a few data frames
containers_2018 = [r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\container-20260331T183127Z-3-001\container\ams__container_2018__202001290000_part_0.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\container-20260331T183127Z-3-001\container\ams__container_2018__202001290000_part_2.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\container-20260331T183127Z-3-001\container\ams__container_2018__202001290000_part_3.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\container-20260331T183127Z-3-002\container\ams__container_2018__202001290000_part_1.csv']
hazmatclass_2018 = [r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\hazmatclass-20260331T183130Z-3-001\hazmatclass\ams__hazmatclass_2018__202001290000_part_0.csv']
header_2018 = [r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\header-20260331T183134Z-3-001\header\ams__header_2018__202001290000_part_3.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\header-20260331T183134Z-3-002\header\ams__header_2018__202001290000_part_2.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\header-20260331T183134Z-3-003\header\ams__header_2018__202001290000_part_1.csv', r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Bronze-Layer\2018\header-20260331T183134Z-3-004\header\ams__header_2018__202001290000_part_0.csv' ]


In [4]:
container_chunks = []
hazmatclass_chunks = []
header_chunks = []

for container in containers_2018:
    for chunk in pd.read_csv(container, chunksize=100000):
        container_chunks.append(chunk)

containerdf = pd.concat(container_chunks, ignore_index=True)

for hazmatclass in hazmatclass_2018:
    for chunk in pd.read_csv(hazmatclass, chunksize=100000):
        hazmatclass_chunks.append(chunk)

hazmatclassdf = pd.concat(hazmatclass_chunks, ignore_index=True)

for header in header_2018:
    for chunk in pd.read_csv(header, chunksize=100000):
        header_chunks.append(chunk)

headerdf = pd.concat(header_chunks, ignore_index=True)

# Statistical Summary of The 2018 Container Numerical Columns
containerdf.head()

C:\Users\CY522SH\AppData\Local\Temp\ipykernel_29320\2923167962.py:18: DtypeWarning: Columns (27,28,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(header, chunksize=100000):
C:\Users\CY522SH\AppData\Local\Temp\ipykernel_29320\2923167962.py:18: DtypeWarning: Columns (17,18,26,27,28,29,30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(header, chunksize=100000):
C:\Users\CY522SH\AppData\Local\Temp\ipykernel_29320\2923167962.py:18: DtypeWarning: Columns (26,27,28,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(header, chunksize=100000):
C:\Users\CY522SH\AppData\Local\Temp\ipykernel_29320\2923167962.py:18: DtypeWarning: Columns (29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(header, chunksize=100000):
C:\Users\CY522SH\AppData\Local\Temp\ipykernel_29320\292

,identifier,container_number,seal_number_1,seal_number_2,equipment_description_code,container_length,container_height,container_width,container_type,load_status,type_of_service
0,201801010,FCIU9250931,EMCCES9186,NaN,Container,4000,906,800,NaN,Loaded,Container Station
1,201801011,EITU1595313,EMCCES9076,NaN,Container,4000,906,802,4EB0,Loaded,Container Yard
2,201801012,FCIU9250931,EMCCES9186,NaN,Container,4000,906,800,NaN,Loaded,Container Station
3,201801013,BMOU5389685,EMCCES8776,NaN,Container,0,0,0,NaN,Loaded,NaN
4,201801014,EMCU5289450,EMCCES8446,NaN,Container,4000,900,800,45R1,Loaded,Container Yard


In [9]:
# Statistical Summary of The 2018 Container Numerical Columns ( container_length, container_width, container_height, container_weight )
containerdf[['container_length', 'container_width', 'container_height']].describe()

,container_length,container_width,container_height
count,3.151239e+07,3.151239e+07,3.151239e+07
mean,3.164939e+03,7.093328e+02,7.662627e+02
std,1.378522e+03,2.542615e+02,2.783135e+02
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.000000e+03,8.000000e+02,8.060000e+02
50%,4.000000e+03,8.000000e+02,9.000000e+02
75%,4.000000e+03,8.000000e+02,9.000000e+02
max,8.600000e+03,1.306000e+03,2.400000e+03


In [6]:
hazmatclassdf.head()

,identifier,container_number,hazmat_sequence_number,hazmat_classification
0,201801011434,OOLU0361510,1,6.1
1,201801011434,OOLU1921748,1,6.1
2,201801011434,OOLU0361844,1,6.1
3,201801011434,OOLU1467825,1,6.1
4,201801011629,ZCSU2770485,1,6.1


In [11]:
# Statistical Summary of The 2018 hazmatclass Numerical Columns ( hazmat_classification)
hazmatclassdf[['hazmat_classification']].describe()

,hazmat_classification
count,16185
unique,1029
top,3
freq,2944


In [7]:
headerdf.head()

,identifier,carrier_code,vessel_country_code,vessel_name,port_of_unlading,estimated_arrival_date,foreign_port_of_lading_qualifier,foreign_port_of_lading,manifest_quantity,manifest_unit,...,secondary_notify_party_2,secondary_notify_party_3,secondary_notify_party_4,secondary_notify_party_5,secondary_notify_party_6,secondary_notify_party_7,secondary_notify_party_8,secondary_notify_party_9,secondary_notify_party_10,actual_arrival_date
0,2018100651992,SEAU,LR,AS VEGA,"Wilmington, North Carolina",2018-10-04,Schedule K Foreign Port,"Puerto Cortes,Honduras",561,BOX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-10-05
1,2018100651993,MANY,MT,X-PRESS MONTE BIANCO,"New York, New York",2018-06-10,Schedule K Foreign Port,"Casablanca,Morocco",1200,CAS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-07-26
2,2018100651994,MANY,MT,X-PRESS MONTE BIANCO,"New York, New York",2018-06-10,Schedule K Foreign Port,"Casablanca,Morocco",2400,CAS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-07-26
3,2018100651995,DSVF,US,YORKTOWN EXPRESS,"Charleston, South Carolina",2018-09-06,Schedule K Foreign Port,"Bremerhaven,Federal Republic of Germany",30,PKG,...,HLCU,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-09-08
4,2018100651996,DSVF,PA,NYK DAEDALUS,"Los Angeles, California",2018-08-17,Schedule K Foreign Port,"Stadersand,Federal Republic of Germany",1,PKG,...,ONEY,SSLL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-08-18


In [13]:
# Statistical Summary of The 2018 Header Numerical Columns (manifest_quantity)
headerdf[['manifest_quantity']].describe()

,manifest_quantity
count,1.990454e+07
mean,1.254892e+03
std,3.995249e+04
min,1.000000e+00
25%,3.000000e+01
50%,2.430000e+02
75%,1.032000e+03
max,1.185142e+08


- Data Cleaning Methodology
    - Errors
        - Invalid container numbers retained but flagged
        - Invalid value/unit combinations
    - Nulls
        - Allowed: optional operational fields (seals, service type)
        - Not allowed: shipment identifier, container number, hazard class
    - Duplicates
        - Removed using business keys:
            - Header: identifier
            - Container: identifier, container_number
            - Hazmat: identifier, container_number, hazmat_sequence_number
    - Outliers
        - Extreme weights/quantities flagged, not removed
    
- Silver Layer Data Quality Standards
    - Enforced schema and data types
    - Referential integrity across header, container and hazmatclass
    - Deterministic deduplication
    - Lifecycle traceablility

- External Sources
    - I am not sure, but I will look at hazard code references, container standards, and Carrier master data possibly for the Gold Layer

## 2 — Proposed Solution

### A — Silver Layer ERD

#### 1 — Header
- **Primary Key (PK):** `identifier`
- **Attributes:**
  - carrier_code
  - vessel_name
  - foreign_port_of_landing
  - port_of_unlading
  - port_of_destination
  - estimated_arrival_date
  - actual_arrival_date
  - record_status_indicator
  - manifest_quantity
  - weight (with unit)

---

#### 2 — Container
- **Primary Key (PK):** (`identifier`, `container_number`)
- **Foreign Key (FK):** `identifier` → Shipment_Header
- **Attributes:**
  - container_length
  - container_height
  - container_width
  - container_type
  - load_status
  - type_of_service
  - seal_number_1
  - seal_number_2
  - equipment_description_code

---

#### 3 — Hazmat_Class
- **Primary Key (PK):** (`identifier`, `container_number`, `hazmat_sequence_number`)
- **Foreign Key (FK):** (`identifier`, `container_number`) → Container
- **Attributes:**
  - hazmat_classification
``


  


  ERD Relationships:
  
    - **Shipment_Header** (1)
        - **Container** (many)
             - **Hazmat_Class** (many)
``

### B — Gold Layer Design

The Gold layer is **denormalized and analytics‑focused**, built from Silver tables to support reporting and decision‑making.

#### 1 — `gold_hazmat_shipments`
**Purpose:** End‑to‑end hazardous shipment analytics

- **Sources:** Shipment_Header, Container, Hazmat_Class  
- **Grain:** One row per container–hazmat combination  
- **Design:** Shipment, container, and hazard attributes flattened  
- **Key fields:** shipment_identifier, vessel_name, carrier_code, arrival_date, port_of_unlading, container_number, container_dimensions, load_status, hazmat_classification  
- **Supports:** port hazard volumes, high‑risk vessel identification, year‑over‑year trends, regulatory reporting  

---

#### 2 — `gold_port_hazmat_summary`
**Purpose:** Port‑level hazard analysis

- **Grain:** One row per port, year, and hazard class  
- **Aggregations:** container_count, hazardous_container_count, dominant_hazard_class  
- **Supports:** port risk profiling and inspection resource prioritization

### Partitioning Strategy

- **Primary partition:** `year`
  - Derived from arrival date or source directory (2018–2020)
  - Aligns with raw CBP delivery structure
  - Optimizes multi‑year queries and data pruning

- **Secondary (logical) filters:**  
  - `port_of_unlading`  
  - `hazmat_classification`  
  - `load_status`

The Silver layer is physically partitioned only by **year** to avoid small‑file and skew issues. Additional filtering and optimization are applied in the Gold layer through denormalized views and aggregations.
``

#### 3 — `Use Case`
- Enable aggregated reporting on hazardous cargo movements through major U.S. ports, broken down by hazard type, carrier, and time period.